In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
X_train = pd.read_csv("X_train.csv")
X_test  = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").values.ravel()
y_test  = pd.read_csv("y_test.csv").values.ravel()

In [7]:
X_train = X_train.drop(columns=["int_rate"])
X_test  = X_test.drop(columns=["int_rate"])

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [11]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline

log_model = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty="l2",
    scoring="roc_auc",
    max_iter=1000,
    n_jobs=-1,
    refit=True
)

clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", log_model)
])

In [12]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb)
])

In [13]:
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['loan_amnt', 'term', 'installment', 'emp_length', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'log_annual_inc'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['purpose', 'home_ownership'], dtype='object'))])),
                ('model',
                 LogisticRegressionCV(cv=5, max_iter=1000, n_jobs=-1,
                                      scoring='roc_auc'))])

In [14]:
xgb_clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['loan_amnt', 'term', 'installment', 'emp_length', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'log_annual_inc'],
      dtype='object')),
                                                 ('cat...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None, min_child_weight=5,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=600, n_jobs=-1,
                               num_parallel_tree=None, random_state=42, ...))])

In [15]:
from sklearn.metrics import roc_auc_score, classification_report

# Predict probabilities
train_probs_clf = clf.predict_proba(X_train)[:, 1]
test_probs_clf  = clf.predict_proba(X_test)[:, 1]

print("Train AUC Logistic:", round(roc_auc_score(y_train, train_probs_clf), 4))
print("Test AUC Logistic:", round(roc_auc_score(y_test, test_probs_clf), 4))

Train AUC Logistic: 0.6934
Test AUC Logistic: 0.6859


In [16]:
from sklearn.metrics import roc_auc_score

train_probs_xgb = xgb_clf.predict_proba(X_train)[:, 1]
test_probs_xgb  = xgb_clf.predict_proba(X_test)[:, 1]

print("Train AUC XGBoost:", round(roc_auc_score(y_train, train_probs_xgb), 4))
print("Test AUC XGBoost:", round(roc_auc_score(y_test, test_probs_xgb), 4))

Train AUC XGBoost: 0.7137
Test AUC XGBoost: 0.7064


In [23]:
import json
from datetime import datetime

logistic_robust_metrics = {
    "model": "Logistic Regression (Robust)",
    "train_auc": round(roc_auc_score(y_train, train_probs_clf), 4),
    "test_auc": round(roc_auc_score(y_test, test_probs_clf), 4),
    "validation_split": "Train: ≤2016 | Test: ≥2017",
    "timestamp": datetime.now().isoformat()
}

with open("../public/logistic_robust_metrics.json", "w") as f:
    json.dump(logistic_robust_metrics, f, indent=4)

print("Saved logistic_robust_metrics.json")

Saved logistic_robust_metrics.json


In [24]:
import json
from datetime import datetime

xgb_robust_metrics = {
    "model": "XGBoost (Robust)",
    "train_auc": round(roc_auc_score(y_train, train_probs_xgb), 4),
    "test_auc": round(roc_auc_score(y_test, test_probs_xgb), 4),
    "validation_split": "Train: ≤2016 | Test: ≥2017",
    "timestamp": datetime.now().isoformat()
}

with open("../public/xgboost_robust_metrics.json", "w") as f:
    json.dump(xgb_robust_metrics, f, indent=4)

print("Saved xgboost_robust_metrics.json")

Saved xgboost_robust_metrics.json
